# Video-LLaVA / MuLER Stage 2 Fine-Tuning on Google Colab (Google Drive Pipeline)

### Architecture Overview
Stage 2 fine-tunes the Multimodal LLM (Vicuna-7B) + Vision Projector on:
- **Image Instruction Tuning**: LLaVA-Instruct 665K (`llava_image_tune_2.zip.001` - `002`, ~67.4 GB)
- **Video Instruction Tuning**: Video-ChatGPT 100K (`videochatgpt_tune_2.zip.001` - `005`, ~160.1 GB)
- **NLP Instruction Tuning**: Pure text QA (`nlp_tune.json`)

### Zero-SSD-Exhaustion Storage Architecture:
Since Colab local SSD (~80-100 GB) cannot hold the uncompressed 260+ GB datasets, all data is **downloaded, streamed, and extracted directly onto persistent Google Drive**.

| Component | Storage Location | Transfer Mechanism |
|-----------|------------------|--------------------|
| **Image Tuning Media** (67.4 GB) | **Google Drive** | 16-channel accelerated staging + direct 7z extraction |
| **Video Tuning Media** (160.1 GB) | **Google Drive** | 16-channel accelerated staging + direct 7z extraction |
| **Annotation JSONs** | **Drive & Local SSD** | Synced for low latency tokenization |
| **Checkpoints** | **Local SSD → Drive** | Saved locally for fast I/O, asynchronously muled to Drive |

## 1. Check GPU Environment

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/Video-LLaVA"
os.makedirs(f"{DRIVE_ROOT}/datasets", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
print(f"Google Drive workspace verified at: {DRIVE_ROOT}")

## 3. Clone Repository & Install Dependencies

In [ ]:
%cd /content
![ ! -d "grad_project" ] && git clone https://github.com/davidrimon2004/grad_project
%cd /content/grad_project
!git pull

# Install training dependencies
!pip install -q --upgrade pip
!pip install -q transformers tokenizers sentencepiece shortuuid accelerate peft bitsandbytes einops einops-exts timm deepspeed huggingface_hub decord gdown av
!pip install -q flash-attn --no-build-isolation || echo "Flash attention skipped; falling back to standard attention."
!apt-get install -y -qq aria2 p7zip-full
!pip install -e .

## 4. Check Storage & Download Fine-Tuning Annotations (~471 MB)

In [ ]:
!python scripts/colab_finetune_muler.py \
    --action status \
    --drive_root /content/drive/MyDrive/Video-LLaVA

## 5. Download Fine-Tuning Datasets Directly to Google Drive
Downloads `llava_image_tune_2.zip` (67.4 GB) and `videochatgpt_tune_2.zip` (160.1 GB) directly into Google Drive with 16-channel acceleration and auto-resume.

In [ ]:
!python scripts/colab_finetune_muler.py \
    --action download \
    --drive_root /content/drive/MyDrive/Video-LLaVA

## 6. Extract Datasets on Google Drive
Extracts the split multi-part archives directly into `datasets/llava_image_tune/` and `datasets/videochatgpt_tune/` without intermediate file overhead.

In [ ]:
!python scripts/colab_finetune_muler.py \
    --action extract \
    --drive_root /content/drive/MyDrive/Video-LLaVA

## 7. Launch Stage 2 Fine-Tuning (LoRA / Full)
Trains on the multimodal datasets with automatic background checkpoint synchronization to Google Drive.

In [ ]:
!python scripts/colab_finetune_muler.py \
    --action train \
    --lora True \
    --num_train_epochs 1.0 \
    --learning_rate 2e-4 \
    --drive_root /content/drive/MyDrive/Video-LLaVA

## 8. Monitor Training with TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/checkpoints